# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Setup

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Token order:
# environment variable -> Colab Secret -> prompt as last resort
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# Exact file/directory structure from the working warehouse notebook.
TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected to FlyRank warehouse.")
print("Development decision point: 2026-03-31")

Connected to FlyRank warehouse.
Development decision point: 2026-03-31


In [4]:
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [5]:
DECISION_DATE = "2026-03-31"

In [6]:
content_state = con.sql(
    f"""
    SELECT
        content_hash_id,

        DATE_DIFF(
            'day',
            CAST(content_created_date AS DATE),
            DATE '{DECISION_DATE}'
        ) AS content_age_days

    FROM {TABLES['dim_content']}

    WHERE CAST(content_created_date AS DATE)
          <= DATE '{DECISION_DATE}'
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
content_update_state = con.sql(
    f"""
    SELECT
        content_hash_id,

        CASE
            WHEN CAST(content_updated_date AS DATE)
                 <= DATE '{DECISION_DATE}'
            THEN DATE_DIFF(
                'day',
                CAST(content_updated_date AS DATE),
                DATE '{DECISION_DATE}'
            )
            ELSE NULL
        END AS days_since_last_update

    FROM {TABLES['dim_content']}

    WHERE CAST(content_created_date AS DATE)
          <= DATE '{DECISION_DATE}'
    """
).df()

In [8]:
content_state = content_state.merge(
    content_update_state,
    on="content_hash_id",
    how="left"
)

In [9]:
historical_features = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_prev30,

        SUM(gsc_clicks) AS clicks_prev30,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN
                SUM(gsc_sum_position)
                / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_prev30

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-03-30'

      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
FEATURES = [
    "imp_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_age_days",
    "days_since_last_update",
]

feature_frame = historical_features.merge(
    content_state,
    on="content_hash_id",
    how="left"
)

feature_frame = feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        *FEATURES
    ]
].copy()

feature_frame.head()

,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30,content_age_days,days_since_last_update
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,76.0,0.0,4.263158,47,<NA>
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10305.0,22.0,8.124017,47,<NA>
2,client_62f4a7e64f5e0096,content_e689bc511192751a,59.0,0.0,5.813559,47,<NA>
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,692.0,1.0,5.819364,47,<NA>
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,14.360000,47,<NA>


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.